# D220 - Hadoop: Getting Started

A command-focused lab for a single-node **Hadoop 3.3.6** cluster on Ubuntu/WSL with **Java 11**. Hive 4.0.1 is installed on the same machine, but this notebook concentrates on HDFS, YARN, and MapReduce operations.
> Run this commands by coping them into Linux cell, not directly from Jupyter notebook. 
> Run this notebook from Jupyter running inside Ubuntu/WSL. Each code cell uses `%%bash`. Run cells in order.

## 1. The three parts we will operate

- **HDFS** stores files across blocks. The NameNode manages metadata; DataNodes store blocks.
- **YARN** manages cluster resources. The ResourceManager schedules work; NodeManagers run containers.
- **MapReduce** processes data. Its JobHistory Server keeps information about completed jobs.

On this training machine all daemons run on one host, but they still have separate processes and responsibilities.

## 2. Confirm the environment

Verify the Linux user, Java version, Hadoop version, and important paths before starting services.

In [ ]:
%%bash
echo "User: $(whoami)"
echo "Host: $(hostname)"
echo "JAVA_HOME=${JAVA_HOME:-not set}"
echo "HADOOP_HOME=${HADOOP_HOME:-not set}"
echo "HADOOP_CONF_DIR=${HADOOP_CONF_DIR:-not set}"
java -version 2>&1 | head -1
hadoop version | head -2

Ask Hadoop for resolved configuration values. These should match the manual setup.

In [ ]:
%%bash
for key in fs.defaultFS hadoop.tmp.dir dfs.namenode.name.dir dfs.datanode.data.dir
do
  printf '%-30s = ' "$key"
  hdfs getconf -confKey "$key"
done

## 3. Start HDFS

`start-dfs.sh` starts the NameNode, DataNode, and SecondaryNameNode listed by the local configuration. It uses SSH even for `localhost`. Do not format the NameNode during a normal startup; formatting creates a new filesystem identity.

In [ ]:
%%bash
start-dfs.sh

## 4. Meet `jps`: Java process IDs

`jps` is included with the JDK. It lists the PID and short class name of Java processes owned by the current user. After HDFS starts, expect `NameNode`, `DataNode`, and `SecondaryNameNode`. The `Jps` row is the command itself.

In [ ]:
%%bash
jps
echo
echo 'Detailed Java process command lines:'
jps -lv

Linux process tools provide more detail. `pgrep` finds matching PIDs and command lines; `ps` shows selected columns. A PID changes whenever a daemon restarts.

In [ ]:
%%bash
pgrep -af 'NameNode|DataNode|ResourceManager|NodeManager|JobHistoryServer' || true
echo
ps -u "$USER" -o pid,ppid,stat,etime,%cpu,%mem,cmd | grep -E 'NameNode|DataNode|ResourceManager|NodeManager|JobHistoryServer' | grep -v grep || true

## 5. HDFS health and basic file commands

The report should show `Live datanodes (1)` on this single-node cluster. Safe mode should normally be `OFF` after startup.

In [ ]:
%%bash
hdfs dfsadmin -report
echo
hdfs dfsadmin -safemode get
echo
hdfs fsck / -summary

HDFS has its own namespace. `hdfs dfs` commands resemble Linux commands, but they operate on HDFS rather than the local disk. Relative paths are resolved below `/user/$USER`.

In [ ]:
%%bash
hdfs dfs -mkdir -p "/user/$USER/d220/input"
printf '%s\n' \
  'hadoop stores large data' \
  'hadoop uses hdfs' \
  'mapreduce runs on yarn' > /tmp/d220_words.txt
hdfs dfs -put -f /tmp/d220_words.txt "/user/$USER/d220/input/"
hdfs dfs -ls -h "/user/$USER/d220/input"
hdfs dfs -cat "/user/$USER/d220/input/d220_words.txt"
hdfs dfs -du -h "/user/$USER/d220"

Useful exploration commands (run individually when needed):

```bash
hdfs dfs -ls /                       # list a directory
hdfs dfs -ls -R /user/$USER          # recursive listing
hdfs dfs -stat '%n %b bytes' PATH     # file name and size
hdfs dfs -get HDFS_PATH LOCAL_PATH    # download from HDFS
hdfs dfs -cp SOURCE DESTINATION       # copy inside HDFS
hdfs dfs -mv SOURCE DESTINATION       # move/rename inside HDFS
hdfs dfs -rm PATH                     # delete a file
hdfs dfs -rm -r DIRECTORY             # delete a directory tree
hdfs dfs -help ls                     # help for one command
```

## 6. Start YARN and the MapReduce History Server

YARN starts the ResourceManager and NodeManager. The history server is separate and must be started explicitly.

In [ ]:
%%bash
start-yarn.sh
mapred --daemon start historyserver
sleep 2
jps

Verify that YARN sees one healthy NodeManager and inspect scheduler/resource information.

In [ ]:
%%bash
yarn node -list -all
echo
yarn node -status "$(yarn node -list -all 2>/dev/null | awk '/RUNNING/{print $1; exit}')" 2>/dev/null || true
echo
yarn queue -status default

## 7. Which PID owns which port?

`ss` lists sockets. `-lntp` means listening, TCP, numeric addresses, with process details. Process details may require `sudo`.

| Component | Default port | Purpose |
|---|---:|---|
| NameNode | 9870 | Web UI |
| DataNode | 9864 | Web UI |
| ResourceManager | 8088 | Web UI |
| NodeManager | 8042 | Web UI |
| JobHistory Server | 19888 | Web UI |
| NameNode | 9000 | HDFS client RPC (configured in this lab) |
| DataNode | 9866 / 9867 | Data transfer / IPC |
| ResourceManager | 8030-8033 | Scheduler, tracker, client, admin RPC |
| JobHistory Server | 10020 | History RPC |

In [ ]:
%%bash
PORTS='9870|9864|8088|8042|19888|9000|9866|9867|8030|8031|8032|8033|10020'
ss -lntp | grep -E ":($PORTS)\b" || true
echo
echo 'Use this outside the notebook if process names are hidden:'
echo "sudo ss -lntp | grep -E ':($PORTS)\b'"

Test the HTTP endpoints without opening a browser. `curl -I` requests response headers only; `200`, `302`, or another HTTP response proves that the port answered.

In [ ]:
%%bash
for url in http://127.0.0.1:9870/ http://127.0.0.1:8088/ http://127.0.0.1:19888/
do
  printf '%-32s ' "$url"
  curl -sS -o /dev/null -w 'HTTP %{http_code}\n' "$url" || echo 'UNREACHABLE'
done

Open these from a browser on the same computer: [NameNode](http://localhost:9870), [ResourceManager](http://localhost:8088), and [JobHistory](http://localhost:19888). The RPC ports are for Hadoop clients, not web browsers.

## 8. Submit a MapReduce job

MapReduce output directories must not already exist. The guarded removal below deletes only this lab's previous output, making the cell repeatable. During execution, watch the ResourceManager UI or run the application-list cell in another terminal.

In [ ]:
%%bash
INPUT="/user/$USER/d220/input"
OUTPUT="/user/$USER/d220/output"
EXAMPLES_JAR="$HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.3.6.jar"
hdfs dfs -rm -r -f "$OUTPUT"
hadoop jar "$EXAMPLES_JAR" wordcount "$INPUT" "$OUTPUT"
echo 'Word-count result:'
hdfs dfs -cat "$OUTPUT/part-r-00000"

## 9. Explore and control YARN applications

An application ID looks like `application_1712345678901_0001`. `RUNNING` jobs appear in the first command; `ALL` includes completed and failed jobs.

In [ ]:
%%bash
echo 'Running applications:'
yarn application -list
echo
echo 'Recent applications in every state:'
yarn application -list -appStates ALL

Use a real application ID in these commands. Status and logs are read-only. Kill permanently stops a running application, so inspect the ID first.

```bash
APP_ID=application_...
yarn application -status "$APP_ID"
yarn logs -applicationId "$APP_ID"
yarn application -kill "$APP_ID"   # only when you intend to stop it
```

To practice killing an application, start a longer example such as `pi` in one terminal, list its ID in another, check status, and then kill it:

```bash
hadoop jar "$HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.3.6.jar" pi 20 100000
yarn application -list
yarn application -status APPLICATION_ID
yarn application -kill APPLICATION_ID
```

## 10. Logs and troubleshooting

Start with daemon presence, node reports, ports, and recent logs. A Java process can exist while its service is still unhealthy, so use more than `jps`.

In [ ]:
%%bash
echo '=== Java daemons ==='
jps
echo '=== HDFS live-node summary ==='
hdfs dfsadmin -report | grep -E 'Live datanodes|Dead datanodes|Name:' || true
echo '=== YARN nodes ==='
yarn node -list -all
echo '=== Recent daemon log files ==='
find "$HOME/hadoop-logs" -maxdepth 1 -type f -printf '%T@ %p\n' 2>/dev/null | sort -nr | head -10 | cut -d' ' -f2-

Search recent log messages when a service fails. Matches are clues, not always the root cause.

In [ ]:
%%bash
grep -RniE 'exception|error|failed|inaccessible|unsupported|could not' "$HOME/hadoop-logs" 2>/dev/null | tail -80 || true

## 11. Clean shutdown

Stop in reverse order: history server, YARN, then HDFS. A clean shutdown protects data and avoids confusing stale processes in the next lab.

In [ ]:
%%bash
mapred --daemon stop historyserver
stop-yarn.sh
stop-dfs.sh
echo 'Remaining Java processes:'
jps

## Command recap

| Goal | Command |
|---|---|
| Start/stop HDFS | `start-dfs.sh` / `stop-dfs.sh` |
| Start/stop YARN | `start-yarn.sh` / `stop-yarn.sh` |
| Start/stop history | `mapred --daemon start historyserver` / `mapred --daemon stop historyserver` |
| List Java daemons | `jps`, `jps -lv` |
| Inspect Linux processes | `pgrep -af PATTERN`, `ps ...` |
| Inspect listening ports | `sudo ss -lntp` |
| HDFS health | `hdfs dfsadmin -report`, `hdfs fsck / -summary` |
| HDFS files | `hdfs dfs -ls`, `-put`, `-get`, `-cat`, `-du`, `-rm` |
| YARN nodes | `yarn node -list -all` |
| List applications | `yarn application -list -appStates ALL` |
| Application status | `yarn application -status APPLICATION_ID` |
| Application logs | `yarn logs -applicationId APPLICATION_ID` |
| Stop an application | `yarn application -kill APPLICATION_ID` |